In [ ]:
import scanpy as sc
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats
import pickle

import decoupler as dc

pd.set_option("display.max_rows", 500)
pd.set_option("display.expand_frame_repr", False)  # запрещаем перенос строк

/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


### read/prep adata

In [2]:
adata = sc.read_h5ad("./data/adata_all_harm_ribo.h5ad")

In [3]:
sp_scores = pd.read_csv(
    "./data/sen_paper/sp_2nd_scores_is_sen_labels.csv",
    index_col=0,
)
adata.obs.drop(columns=["sp_score_sigs"], inplace=True, errors="ignore")
adata.obs = adata.obs.join(sp_scores, how="left")
adata = adata[adata.obs["is_sen_GMMk2_median_ct"].notna()].copy()

In [ ]:
adata.obs["sen_group"] = pd.NA

mask_sc = adata.obs["is_sen_GMMk2_median_ct"]
adata.obs.loc[mask_sc, "sen_group"] = "Sc"
adata.obs.loc[~mask_sc, "sen_group"] = "NSc"

In [5]:
adata

AnnData object with n_obs × n_vars = 876940 × 43087
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final', 'Age_bin', 'sp_score_sigs_dw', 'centered_median', 'is_sen_GMMk2_median_ct', 'sen_group'
    uns: 'Tissue_global_colors'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [6]:
adata.obs.sen_group.value_counts()

sen_group
NSc    716011
Sc     160929
Name: count, dtype: int64

### Create PBs

In [ ]:
grouped = (
    adata.obs[adata.obs.sen_group.notna()]
    .groupby(
        ["Tissue_global", "cell_type_final", "GSE_id", "GSM_id", "sen_group"],
        observed=True,
    )
    .size()
    .unstack(fill_value=0)  # разделяем Sc и NSc в отдельные колонки
)

grouped["NSc_to_Sc_ratio"] = grouped["NSc"] / grouped["Sc"].replace(0, pd.NA)

result = grouped.reset_index()

result = result.rename(columns={"Sc": "Sc_cells", "NSc": "NSc_cells"})

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print(result)

In [ ]:
adata_filt = adata[adata.obs["sen_group"].notna()].copy()

gsm_counts = (
    adata_filt.obs.groupby(
        ["Tissue_global", "cell_type_final", "GSM_id", "sen_group"], observed=True
    )
    .size()
    .unstack(fill_value=0)
)
# 2. фильтр GSM по минимуму клеток
valid_gsms = gsm_counts[(gsm_counts["Sc"] >= 10) & (gsm_counts["NSc"] >= 10)].index
group_keys = pd.MultiIndex.from_arrays(
    [
        adata_filt.obs["Tissue_global"],
        adata_filt.obs["cell_type_final"],
        adata_filt.obs["GSM_id"],
        adata_filt.obs["sen_group"],
    ]
)

adata_filt = adata_filt[group_keys.isin(valid_gsms)].copy()

# 3. фильтр cell type: >=3 GSM на (tissue, cell_type)
ct_gsm_counts = adata_filt.obs.groupby(
    ["Tissue_global", "cell_type_final"], observed=True
)["GSM_id"].nunique()
valid_ct = ct_gsm_counts[ct_gsm_counts >= 3].index

ct_key = pd.MultiIndex.from_arrays(
    [adata_filt.obs["Tissue_global"], adata_filt.obs["cell_type_final"]]
)
adata_filt = adata_filt[ct_key.isin(valid_ct)].copy()


In [10]:
adata_filt.obs["Age"] = adata_filt.obs["Age"].astype(int)
adata_pb = sc.get.aggregate(
    adata_filt,
    by=["Tissue_global", "cell_type_final", "GSE_id", "GSM_id", "Age", "sen_group"],
    func="sum",
)

In [11]:
adata_pb

AnnData object with n_obs × n_vars = 1434 × 43087
    obs: 'Tissue_global', 'cell_type_final', 'GSE_id', 'GSM_id', 'Age', 'sen_group'
    layers: 'sum'

In [12]:
adata_pb.X = adata_pb.layers["sum"].copy()

In [13]:
adata_pb.obs.head(3)

,Tissue_global,cell_type_final,GSE_id,GSM_id,Age,sen_group
Colon_B cell_GSE214695_GSM6614348_61_NSc,Colon,B cell,GSE214695,GSM6614348,61,NSc
Colon_B cell_GSE214695_GSM6614348_61_Sc,Colon,B cell,GSE214695,GSM6614348,61,Sc
Colon_B cell_GSE214695_GSM6614350_68_NSc,Colon,B cell,GSE214695,GSM6614350,68,NSc


In [ ]:
adata_pb.write("./data/sen_paper/2_adata_pb_NScFull.h5ad")

### DEA edgePython

In [ ]:
# import sys
# !{sys.executable} -m pip install edgepython

In [15]:
import edgepython as ep
import patsy

In [ ]:
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

pandas2ri.activate()

base = importr("base")
edgeR = importr("edgeR")
stats = importr("stats")

counts_df = adata_pb.to_df().T
groups = adata_pb.obs.groupby(["Tissue_global", "cell_type_final"], observed=True)
all_results = []

2026-06-12 14:54:32 | [INFO] cffi mode is CFFI_MODE.ANY
2026-06-12 14:54:32 | [INFO] R home found: /usr/lib/R
2026-06-12 14:54:32 | [INFO] R library path: /usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/opt/conda/lib/server
2026-06-12 14:54:32 | [INFO] LD_LIBRARY_PATH: /usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/opt/conda/lib/server
2026-06-12 14:54:32 | [INFO] Default options to initialize R: rpy2, --quiet, --no-save
2026-06-12 14:54:32 | [INFO] R is already initialized. No need to initialize.


In [ ]:
r_version = robjects.r.packageVersion("edgeR")

version_str = str(r_version[0])
print(f"Версия edgeR: {version_str}")

Версия edgeR: [1]  3 36  0



In [ ]:
for n, ((t, ct), sub_meta) in enumerate(groups):
    # Проверяем, что в группе есть и Sc, и NSc образцы
    if len(sub_meta["sen_group"].unique()) < 2:
        print(f"Пропуск группы {t} {ct}: недостаточно условий для сравнения.")
        continue

    # Вытаскиваем соответствующие колонки из матрицы подсчетов
    sub_counts = counts_df[sub_meta.index]
    sub_meta = sub_meta.copy()  # защищает от SettingWithCopyWarning
    sub_meta["GSM_id"] = sub_meta["GSM_id"].cat.remove_unused_categories()

    # Временный DGEList ТОЛЬКО для того, чтобы запустить фильтрацию генов
    y_temp = ep.make_dgelist(
        counts=sub_counts.to_numpy(), group=sub_meta["sen_group"].values
    )
    # 3. Получаем маску низкоэкспрессированных генов (edgePython вернет True/False для каждой строки)
    kept_genes_mask = ep.filter_by_expr(y_temp)
    # 4. ФИЛЬТРУЕМ ИСХОДНЫЙ ДАТАФРЕЙМ средствами Pandas (это работает на 100% стабильно)
    sub_counts = sub_counts.iloc[kept_genes_mask]
    # 5. Создаем ФИНАЛЬНЫЙ DGEList уже из чистых, отфильтрованных данных
    y = ep.make_dgelist(
        counts=sub_counts.to_numpy(), group=sub_meta["sen_group"].values
    )

    # Нормализация TMM (Trimmed Mean of M-values)
    y = ep.calc_norm_factors(y)

    # Строим парную дизайн-матрицу: GSM_id блокирует донора, sen_group оценивает сенесцентность
    # Формула автоматически превратит категориальные переменные в dummy-колонки
    formula_str = "~ GSM_id + sen_group"
    design_df = patsy.dmatrix(
        formula_str, data=sub_meta, return_type="dataframe"
    )  # Строим DataFrame дизайна

    col_names = list(design_df.columns)

    # Ищем индекс целевого контраста (сенесцентные клетки)
    target_pattern = "[T.Sc]"
    coef_idx = [i for i, name in enumerate(col_names) if target_pattern in name][0]

    # -------- Передача данных в оригинальный edgeR (R) --------------

    # 1. Конвертируем Python/Pandas объекты в R объекты
    r_counts = pandas2ri.py2rpy(sub_counts)
    r_design = robjects.r["as.matrix"](
        pandas2ri.py2rpy(design_df)
    )  # функции тут вызываются через квадратные скобки

    # 2. Запускаем оригинальный сверхбыстрый пайплайн edgeR на C++
    dge = edgeR.DGEList(counts=r_counts)
    dge = edgeR.calcNormFactors(dge)

    # Оценка дисперсии (в R это займет полсекунды)
    dge = edgeR.estimateDisp(dge, r_design)

    # Квази-правдоподобное фитирование (QL GLM)
    fit = edgeR.glmQLFit(dge, r_design)

    # Тестируем наш коэффициент (в R индексы начинаются с 1, поэтому coef_idx + 1)
    r_res = edgeR.glmQLFTest(fit, coef=int(coef_idx + 1))

    # 3. Извлекаем результаты в формате Pandas DataFrame
    # topTags вытаскивает все гены (n=Inf)
    r_top_tags = edgeR.topTags(r_res, n=float("inf"))
    top_table_r = robjects.r["as.data.frame"](r_top_tags.rx2("table"))
    top_table = pandas2ri.rpy2py(top_table_r)

    top_table["Tissue"] = t
    top_table["cell_type"] = ct

    # Сохраняем результат
    all_results.append(top_table)

    print(
        f"Группа {t} {ct} завершена ({len(sub_counts)} генов, {sub_counts.shape[1]} образцов)"
    )


In [18]:
top_table.head(1)

,logFC,logCPM,F,PValue,FDR,Tissue,cell_type
BCL11B,2.570753,2.84241,219.913529,4.252828e-14,5.577159e-10,Skin,VE


In [19]:
top_table.shape

(13114, 7)

In [ ]:
deg_df = pd.concat(all_results).reset_index(names="gene_name")
deg_df.to_csv("./data/sen_paper/2_pbDEGs_edgeR_NScFull.csv")

In [30]:
deg_filt = deg_df[(np.abs((deg_df.logFC)) > 0.5) & (deg_df.FDR < 0.05)]
deg_filt.shape

(26770, 8)

In [35]:
import gseapy as gp

In [ ]:
# Оставляем только те ct, чьи группы целиком имеют размер >= 5
deg_filt = deg_filt.groupby(["Tissue", "cell_type"], observed=True).filter(
    lambda x: len(x) >= 5
)

In [68]:
deg_df.head()

,gene_name,logFC,logCPM,F,PValue,FDR,Tissue,cell_type
0,MCOLN2,8.531128,6.738176,621.835134,6.380477e-15,1.955616e-11,Colon,B cell
1,TOX,2.008261,7.930047,126.807232,1.632478e-12,2.501772e-09,Colon,B cell
2,PRPSAP2,1.959064,7.334242,109.729473,9.841678e-12,1.005491e-08,Colon,B cell
3,SOX5,2.467394,8.720462,104.922411,1.698185e-11,1.247251e-08,Colon,B cell
4,ENSG00000225885,2.642697,8.036843,103.366315,2.034668e-11,1.247251e-08,Colon,B cell
